In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kgan31/doha-data/kavitakosh_dohas_split.csv


In [3]:
# """
# TF-IDF Retrieval Baseline for Hindi Doha Dataset
# =================================================
# Given a query (theme/context in Hindi or English), retrieves the most
# relevant doha(s) from the dataset using TF-IDF cosine similarity.

# Usage:
#     python tfidf_doha_baseline.py

# Requirements:
#     pip install pandas scikit-learn indic-nlp-library
#     (falls back to basic Unicode tokenization if indic-nlp not available)
# """

# import pandas as pd
# import numpy as np
# import re
# import warnings
# warnings.filterwarnings("ignore")

# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity

# # ─────────────────────────────────────────────
# # 1. LOAD DATASET
# # ─────────────────────────────────────────────

# def load_dataset(csv_path: str) -> pd.DataFrame:
#     """Load and clean the doha CSV dataset."""
#     df = pd.read_csv(csv_path)
    
#     # Normalize column names
#     df.columns = [c.strip() for c in df.columns]
    
#     # Drop rows with missing dohas
#     df = df.dropna(subset=["Doha"]).reset_index(drop=True)
    
#     # Clean doha text: remove verse numbers like ।।3।। and extra whitespace
#     df["Doha_clean"] = df["Doha"].apply(clean_doha)
    
#     # Build a retrieval document: combine title + doha for richer matching
#     df["retrieval_doc"] = df.apply(
#         lambda r: f"{r.get('Title', '')} {r.get('Author', '')} {r['Doha_clean']}",
#         axis=1
#     )
    
#     print(f"✅ Loaded {len(df)} dohas from {csv_path}")
#     return df


# def clean_doha(text: str) -> str:
#     """Remove verse markers and normalize whitespace."""
#     if not isinstance(text, str):
#         return ""
#     # Remove ।।number।। patterns
#     text = re.sub(r"।।\s*\d+\s*।।", "", text)
#     # Normalize newlines and spaces
#     text = re.sub(r"\s+", " ", text).strip()
#     return text


# # ─────────────────────────────────────────────
# # 2. HINDI TOKENIZER
# # ─────────────────────────────────────────────

# def hindi_tokenizer(text: str) -> list:
#     """
#     Simple Unicode-based tokenizer for Hindi/Braj text.
#     Splits on whitespace and punctuation, preserving Devanagari characters.
#     Falls back gracefully if indic-nlp is not installed.
#     """
#     # Remove Devanagari dandas and punctuation
#     text = re.sub(r"[।॥,!?\"'()\[\]{}]", " ", text)
#     tokens = text.split()
#     # Filter empty tokens and tokens with only non-Devanagari chars
#     tokens = [t for t in tokens if re.search(r"[\u0900-\u097F]", t)]
#     return tokens


# def build_tfidf_index(df: pd.DataFrame):
#     """
#     Build a TF-IDF matrix over the doha corpus.
#     Returns the fitted vectorizer and the TF-IDF matrix.
#     """
#     vectorizer = TfidfVectorizer(
#         tokenizer=hindi_tokenizer,
#         token_pattern=None,        # Use custom tokenizer
#         analyzer="word",
#         ngram_range=(1, 2),        # Unigrams + bigrams
#         min_df=1,                  # Include rare terms (small corpus)
#         max_df=0.95,               # Ignore near-universal terms
#         sublinear_tf=True,         # Apply log normalization to TF
#     )
    
#     tfidf_matrix = vectorizer.fit_transform(df["retrieval_doc"])
#     print(f"✅ TF-IDF index built — vocab size: {len(vectorizer.vocabulary_)}, "
#           f"matrix shape: {tfidf_matrix.shape}")
#     return vectorizer, tfidf_matrix


# # ─────────────────────────────────────────────
# # 3. RETRIEVAL FUNCTION
# # ─────────────────────────────────────────────

# def retrieve_dohas(
#     query: str,
#     df: pd.DataFrame,
#     vectorizer,
#     tfidf_matrix,
#     top_k: int = 5
# ) -> pd.DataFrame:
#     """
#     Given a Hindi/English query string, return the top-k most relevant dohas.

#     Args:
#         query    : Theme or context string (e.g., "प्रेम और विरह" or "nature beauty")
#         df       : The doha dataframe
#         vectorizer: Fitted TF-IDF vectorizer
#         tfidf_matrix: Precomputed TF-IDF matrix
#         top_k    : Number of results to return

#     Returns:
#         DataFrame of top-k results with similarity scores
#     """
#     # Vectorize the query
#     query_vec = vectorizer.transform([query])
    
#     # Compute cosine similarity between query and all dohas
#     similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
#     # Get top-k indices
#     top_indices = similarities.argsort()[::-1][:top_k]
    
#     results = df.iloc[top_indices].copy()
#     results["similarity_score"] = similarities[top_indices]
#     results["rank"] = range(1, len(results) + 1)
    
#     return results[["rank", "similarity_score", "Title", "Author", 
#                      "Doha_Number", "Doha", "URL"]].reset_index(drop=True)


# # ─────────────────────────────────────────────
# # 4. EVALUATION
# # ─────────────────────────────────────────────

# def evaluate_retrieval(
#     test_queries: list,
#     df: pd.DataFrame,
#     vectorizer,
#     tfidf_matrix,
#     top_k: int = 5
# ):
#     """
#     Evaluate the TF-IDF retrieval system on a list of test queries.

#     Args:
#         test_queries: List of dicts with keys 'query' and optionally 'relevant_title'
#                       e.g. [{"query": "केश सौंदर्य", "relevant_title": "अंग दर्पण"}]
#     """
#     print("\n" + "="*60)
#     print("EVALUATION RESULTS")
#     print("="*60)
    
#     hits_at_1 = 0
#     hits_at_k = 0
#     mrr_scores = []
    
#     for item in test_queries:
#         query = item["query"]
#         relevant_title = item.get("relevant_title", None)
        
#         results = retrieve_dohas(query, df, vectorizer, tfidf_matrix, top_k=top_k)
        
#         print(f"\nQuery: '{query}'")
#         print(f"  Top result: [{results.iloc[0]['similarity_score']:.3f}] "
#               f"{results.iloc[0]['Author']} — {results.iloc[0]['Doha'][:60]}...")
        
#         if relevant_title:
#             # Check if relevant title appears in top-k
#             retrieved_titles = results["Title"].tolist()
#             rank = None
#             for i, title in enumerate(retrieved_titles):
#                 if relevant_title in str(title):
#                     rank = i + 1
#                     break
            
#             if rank == 1:
#                 hits_at_1 += 1
#             if rank is not None:
#                 hits_at_k += 1
#                 mrr_scores.append(1.0 / rank)
#                 print(f"  ✅ Relevant doha found at rank {rank}")
#             else:
#                 mrr_scores.append(0.0)
#                 print(f"  ❌ Relevant doha NOT found in top-{top_k}")
    
#     n = len(test_queries)
#     if n > 0:
#         print("\n" + "-"*40)
#         print(f"Hit@1  : {hits_at_1}/{n} = {hits_at_1/n:.2%}")
#         print(f"Hit@{top_k}  : {hits_at_k}/{n} = {hits_at_k/n:.2%}")
#         if mrr_scores:
#             print(f"MRR    : {np.mean(mrr_scores):.3f}")
#         print("-"*40)


# # ─────────────────────────────────────────────
# # 5. DEMO / MAIN
# # ─────────────────────────────────────────────

# def pretty_print_results(results: pd.DataFrame, query: str):
#     """Pretty print retrieval results."""
#     print(f"\n{'='*60}")
#     print(f"Query: \"{query}\"")
#     print(f"{'='*60}")
#     for _, row in results.iterrows():
#         print(f"\nRank {int(row['rank'])}  |  Score: {row['similarity_score']:.4f}")
#         print(f"  Author : {row['Author']}")
#         print(f"  Title  : {row['Title']}")
#         print(f"  Doha   :\n    {row['Doha']}")
#         print(f"  URL    : {row['URL']}")
#         print("-"*50)


# if __name__ == "__main__":
#     import sys

#     # ── Path to your CSV ──────────────────────────────
#     CSV_PATH = "/kaggle/input/datasets/kgan31/doha-data/kavitakosh_dohas_split.csv"   # ← change to your actual path
#     # ─────────────────────────────────────────────────

#     # Load & index
#     df = load_dataset(CSV_PATH)
#     vectorizer, tfidf_matrix = build_tfidf_index(df)

#     # ── Demo queries ──────────────────────────────────
#     demo_queries = [
#         "केश सौंदर्य बेनी",          # Hair beauty
#         "मोर पच्छ सिर",              # Peacock feather
#         "नायिका प्रेम विरह",         # Heroine love separation
#         "सर्प अहि बेनी",             # Snake-like braid
#     ]

#     for q in demo_queries:
#         results = retrieve_dohas(q, df, vectorizer, tfidf_matrix, top_k=3)
#         pretty_print_results(results, q)

#     # ── Evaluation (add your labeled test pairs here) ─
#     # Format: {"query": "...", "relevant_title": "..."}
#     # relevant_title is a substring match against the Title column
#     test_queries = [
#         {"query": "केश सौंदर्य बेनी",    "relevant_title": "अंग दर्पण"},
#         {"query": "मोर पच्छ सिर",        "relevant_title": "अंग दर्पण"},
#         {"query": "नायिका प्रेम विरह",   "relevant_title": "अंग दर्पण"},
#     ]

#     evaluate_retrieval(test_queries, df, vectorizer, tfidf_matrix, top_k=5)

#     # ── Interactive mode ──────────────────────────────
#     print("\n\n--- Interactive Retrieval (type 'quit' to exit) ---")
#     while True:
#         query = input("\nEnter query (Hindi/English): ").strip()
#         if query.lower() in ("quit", "exit", "q"):
#             break
#         if not query:
#             continue
#         top_k = input("How many results? [default 5]: ").strip()
#         top_k = int(top_k) if top_k.isdigit() else 5
#         results = retrieve_dohas(query, df, vectorizer, tfidf_matrix, top_k=top_k)
#         pretty_print_results(results, query)

In [4]:
!pip install indic-nlp-library

# Download indic_nlp_resources
!git clone https://github.com/anoopkunchukuttan/indic_nlp_resources /kaggle/working/indic_nlp_resources

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 61.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 7.4 MB/s eta 0:00:00
Cloning into '/kaggle/working/indic_nlp_resources'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 139 (delta 2), reused 2 (delta 0), pack-reused 126 (from 1)
Receiving objects: 100% (139/139), 149.77 MiB | 46.25 MiB/s, done.
Resolving deltas: 100% (53/53), done.


In [5]:
# In setup_indic_nlp call:
INDIC_NLP_RESOURCES_PATH = "/kaggle/working/indic_nlp_resources"

# In CSV path:
CSV_PATH = "/kaggle/input/datasets/kgan31/doha-data/kavitakosh_dohas_split.csv"

In [6]:
# """
# TF-IDF Retrieval Baseline for Hindi Doha Dataset
# =================================================
# Given a query (theme/context in Hindi or English), retrieves the most
# relevant doha(s) from the dataset using TF-IDF cosine similarity.

# Usage:
#     python tfidf_doha_baseline.py

# Requirements:
#     pip install pandas scikit-learn
#     pip install indic-nlp-library
    
#     Download indic_nlp_resources from:
#     https://github.com/anoopkunchukuttan/indic_nlp_resources
#     Then set INDIC_NLP_RESOURCES env variable or pass the path to setup_indic_nlp() below.
# """

# import os
# import pandas as pd
# import numpy as np
# import re
# import warnings
# warnings.filterwarnings("ignore")

# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity

# # ─────────────────────────────────────────────
# # INDIC NLP SETUP
# # ─────────────────────────────────────────────

# def setup_indic_nlp(resources_path: str = None):
#     """
#     Initialize the indic-nlp-library.

#     Args:
#         resources_path: Path to the cloned indic_nlp_resources folder.
#                         If None, reads from INDIC_NLP_RESOURCES env variable.

#     Download resources:
#         git clone https://github.com/anoopkunchukuttan/indic_nlp_resources
#     """
#     from indicnlp import common as indic_common
#     from indicnlp import loader as indic_loader

#     path = resources_path or os.environ.get("INDIC_NLP_RESOURCES", "")
#     if not path:
#         raise ValueError(
#             "indic_nlp_resources path not set.\n"
#             "Either pass resources_path= to setup_indic_nlp() or set the "
#             "INDIC_NLP_RESOURCES environment variable.\n"
#             "Download: https://github.com/anoopkunchukuttan/indic_nlp_resources"
#         )
#     indic_common.set_resources_path(path)
#     indic_loader.load()
#     print(f"✅ indic-nlp-library initialised (resources: {path})")


# # ─────────────────────────────────────────────
# # 1. LOAD DATASET
# # ─────────────────────────────────────────────

# def load_dataset(csv_path: str) -> pd.DataFrame:
#     """Load and clean the doha CSV dataset."""
#     df = pd.read_csv(csv_path)
    
#     # Normalize column names
#     df.columns = [c.strip() for c in df.columns]
    
#     # Drop rows with missing dohas
#     df = df.dropna(subset=["Doha"]).reset_index(drop=True)
    
#     # Clean doha text: remove verse numbers like ।।3।। and extra whitespace
#     df["Doha_clean"] = df["Doha"].apply(clean_doha)
    
#     # Build a retrieval document: combine title + doha for richer matching
#     df["retrieval_doc"] = df.apply(
#         lambda r: f"{r.get('Title', '')} {r.get('Author', '')} {r['Doha_clean']}",
#         axis=1
#     )
    
#     print(f"✅ Loaded {len(df)} dohas from {csv_path}")
#     return df


# def clean_doha(text: str) -> str:
#     """Remove verse markers and normalize whitespace."""
#     if not isinstance(text, str):
#         return ""
#     # Remove ।।number।। patterns
#     text = re.sub(r"।।\s*\d+\s*।।", "", text)
#     # Normalize newlines and spaces
#     text = re.sub(r"\s+", " ", text).strip()
#     return text


# # ─────────────────────────────────────────────
# # 2. HINDI TOKENIZER (indic-nlp-library)
# # ─────────────────────────────────────────────

# def hindi_tokenizer(text: str) -> list:
#     """
#     Tokenizer for Hindi/Braj text using indic-nlp-library.

#     indic-nlp provides language-aware tokenization for Devanagari script:
#       - Handles Hindi punctuation (danda ।, double danda ॥) correctly
#       - Splits clitics and postpositions (e.g., "घर+में" → ["घर", "में"])
#       - Works with Braj Bhasha (older Hindi dialect used in dohas)
#       - Language code 'hi' covers standard Hindi + Braj Devanagari

#     Falls back to regex-based Unicode tokenization if indic-nlp is
#     unavailable, so the rest of the pipeline always works.
#     """
#     try:
#         from indicnlp.tokenize import indic_tokenize
#         # indic_tokenize.trivial_tokenize handles Devanagari punctuation
#         # and whitespace splitting in a language-aware way
#         tokens = indic_tokenize.trivial_tokenize(text, lang="hi")
#         # Filter out pure punctuation tokens (dandas, digits, symbols)
#         tokens = [
#             t for t in tokens
#             if re.search(r"[\u0900-\u097F]", t)   # must contain Devanagari
#         ]
#         return tokens

#     except ImportError:
#         # Graceful fallback: plain Unicode split
#         text = re.sub(r"[।॥,!?\"'()\[\]{}]", " ", text)
#         tokens = text.split()
#         tokens = [t for t in tokens if re.search(r"[\u0900-\u097F]", t)]
#         return tokens


# def build_tfidf_index(df: pd.DataFrame):
#     """
#     Build a TF-IDF matrix over the doha corpus.
#     Returns the fitted vectorizer and the TF-IDF matrix.
#     """
#     vectorizer = TfidfVectorizer(
#         tokenizer=hindi_tokenizer,
#         token_pattern=None,        # Use custom tokenizer
#         analyzer="word",
#         ngram_range=(1, 2),        # Unigrams + bigrams
#         min_df=1,                  # Include rare terms (small corpus)
#         max_df=0.95,               # Ignore near-universal terms
#         sublinear_tf=True,         # Apply log normalization to TF
#     )
    
#     tfidf_matrix = vectorizer.fit_transform(df["retrieval_doc"])
#     print(f"✅ TF-IDF index built — vocab size: {len(vectorizer.vocabulary_)}, "
#           f"matrix shape: {tfidf_matrix.shape}")
#     return vectorizer, tfidf_matrix


# # ─────────────────────────────────────────────
# # 3. RETRIEVAL FUNCTION
# # ─────────────────────────────────────────────

# def retrieve_dohas(
#     query: str,
#     df: pd.DataFrame,
#     vectorizer,
#     tfidf_matrix,
#     top_k: int = 5
# ) -> pd.DataFrame:
#     """
#     Given a Hindi/English query string, return the top-k most relevant dohas.

#     Args:
#         query    : Theme or context string (e.g., "प्रेम और विरह" or "nature beauty")
#         df       : The doha dataframe
#         vectorizer: Fitted TF-IDF vectorizer
#         tfidf_matrix: Precomputed TF-IDF matrix
#         top_k    : Number of results to return

#     Returns:
#         DataFrame of top-k results with similarity scores
#     """
#     # Vectorize the query
#     query_vec = vectorizer.transform([query])
    
#     # Compute cosine similarity between query and all dohas
#     similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
#     # Get top-k indices
#     top_indices = similarities.argsort()[::-1][:top_k]
    
#     results = df.iloc[top_indices].copy()
#     results["similarity_score"] = similarities[top_indices]
#     results["rank"] = range(1, len(results) + 1)
    
#     return results[["rank", "similarity_score", "Title", "Author", 
#                      "Doha_Number", "Doha", "URL"]].reset_index(drop=True)


# # ─────────────────────────────────────────────
# # 4. EVALUATION
# # ─────────────────────────────────────────────

# def evaluate_retrieval(
#     test_queries: list,
#     df: pd.DataFrame,
#     vectorizer,
#     tfidf_matrix,
#     top_k: int = 5
# ):
#     """
#     Evaluate the TF-IDF retrieval system on a list of test queries.

#     Args:
#         test_queries: List of dicts with keys 'query' and optionally 'relevant_title'
#                       e.g. [{"query": "केश सौंदर्य", "relevant_title": "अंग दर्पण"}]
#     """
#     print("\n" + "="*60)
#     print("EVALUATION RESULTS")
#     print("="*60)
    
#     hits_at_1 = 0
#     hits_at_k = 0
#     mrr_scores = []
    
#     for item in test_queries:
#         query = item["query"]
#         relevant_title = item.get("relevant_title", None)
        
#         results = retrieve_dohas(query, df, vectorizer, tfidf_matrix, top_k=top_k)
        
#         print(f"\nQuery: '{query}'")
#         print(f"  Top result: [{results.iloc[0]['similarity_score']:.3f}] "
#               f"{results.iloc[0]['Author']} — {results.iloc[0]['Doha'][:60]}...")
        
#         if relevant_title:
#             # Check if relevant title appears in top-k
#             retrieved_titles = results["Title"].tolist()
#             rank = None
#             for i, title in enumerate(retrieved_titles):
#                 if relevant_title in str(title):
#                     rank = i + 1
#                     break
            
#             if rank == 1:
#                 hits_at_1 += 1
#             if rank is not None:
#                 hits_at_k += 1
#                 mrr_scores.append(1.0 / rank)
#                 print(f"  ✅ Relevant doha found at rank {rank}")
#             else:
#                 mrr_scores.append(0.0)
#                 print(f"  ❌ Relevant doha NOT found in top-{top_k}")
    
#     n = len(test_queries)
#     if n > 0:
#         print("\n" + "-"*40)
#         print(f"Hit@1  : {hits_at_1}/{n} = {hits_at_1/n:.2%}")
#         print(f"Hit@{top_k}  : {hits_at_k}/{n} = {hits_at_k/n:.2%}")
#         if mrr_scores:
#             print(f"MRR    : {np.mean(mrr_scores):.3f}")
#         print("-"*40)


# # ─────────────────────────────────────────────
# # 5. DEMO / MAIN
# # ─────────────────────────────────────────────

# def pretty_print_results(results: pd.DataFrame, query: str):
#     """Pretty print retrieval results."""
#     print(f"\n{'='*60}")
#     print(f"Query: \"{query}\"")
#     print(f"{'='*60}")
#     for _, row in results.iterrows():
#         print(f"\nRank {int(row['rank'])}  |  Score: {row['similarity_score']:.4f}")
#         print(f"  Author : {row['Author']}")
#         print(f"  Title  : {row['Title']}")
#         print(f"  Doha   :\n    {row['Doha']}")
#         print(f"  URL    : {row['URL']}")
#         print("-"*50)


# if __name__ == "__main__":
#     import sys

#     # ── indic-nlp setup ───────────────────────────────
#     # Set this to the path where you cloned indic_nlp_resources, OR
#     # export INDIC_NLP_RESOURCES=/path/to/indic_nlp_resources in your shell
#     # INDIC_NLP_RESOURCES_PATH = ""   # ← set your path here, or leave "" to use env var
#     setup_indic_nlp(INDIC_NLP_RESOURCES_PATH or None)

#     # ── Path to your CSV ──────────────────────────────
#     # CSV_PATH = "doha_dataset.csv"   # ← change to your actual path
#     # ─────────────────────────────────────────────────

#     # Load & index
#     df = load_dataset(CSV_PATH)
#     vectorizer, tfidf_matrix = build_tfidf_index(df)

#     # ── Demo queries ──────────────────────────────────
#     demo_queries = [
#         "केश सौंदर्य बेनी",          # Hair beauty
#         "मोर पच्छ सिर",              # Peacock feather
#         "नायिका प्रेम विरह",         # Heroine love separation
#         "सर्प अहि बेनी",             # Snake-like braid
#     ]

#     for q in demo_queries:
#         results = retrieve_dohas(q, df, vectorizer, tfidf_matrix, top_k=3)
#         pretty_print_results(results, q)

#     # ── Evaluation (add your labeled test pairs here) ─
#     # Format: {"query": "...", "relevant_title": "..."}
#     # relevant_title is a substring match against the Title column
#     test_queries = [
#         {"query": "केश सौंदर्य बेनी",    "relevant_title": "अंग दर्पण"},
#         {"query": "मोर पच्छ सिर",        "relevant_title": "अंग दर्पण"},
#         {"query": "नायिका प्रेम विरह",   "relevant_title": "अंग दर्पण"},
#     ]

#     evaluate_retrieval(test_queries, df, vectorizer, tfidf_matrix, top_k=5)

#     # ── Interactive mode ──────────────────────────────
#     print("\n\n--- Interactive Retrieval (type 'quit' to exit) ---")
#     while True:
#         query = input("\nEnter query (Hindi/English): ").strip()
#         if query.lower() in ("quit", "exit", "q"):
#             break
#         if not query:
#             continue
#         top_k = input("How many results? [default 5]: ").strip()
#         top_k = int(top_k) if top_k.isdigit() else 5
#         results = retrieve_dohas(query, df, vectorizer, tfidf_matrix, top_k=top_k)
#         pretty_print_results(results, query)

In [7]:
!pip install torch nltk
SAVE_DIR = "/kaggle/working"

In [8]:
"""
Character-Level LSTM for Hindi Doha Generation
================================================
Trains a character-level LSTM on your doha dataset, then evaluates using:
  - Perplexity  (language modelling quality)
  - BLEU Score  (n-gram overlap with reference dohas)
  - Sample generation (qualitative check)

Kaggle setup:
    !pip install torch nltk
    CSV_PATH  = "/kaggle/input/doha-dataset/doha_dataset.csv"
    SAVE_DIR  = "/kaggle/working"

Local setup:
    pip install torch nltk pandas numpy
"""

import os
import re
import math
import random
import time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction, corpus_bleu
import nltk
nltk.download("punkt", quiet=True)

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG  — change these as needed
# ─────────────────────────────────────────────────────────────────────────────

# CSV_PATH   = "doha_dataset.csv"          # ← your CSV path
# SAVE_DIR   = ""                         # ← where to save checkpoints

# Model hyperparameters
EMBED_DIM   = 64       # Character embedding size
HIDDEN_DIM  = 256       # LSTM hidden units
NUM_LAYERS  = 2         # Stacked LSTM layers
DROPOUT     = 0.5      # Dropout between LSTM layers

# Training hyperparameters
SEQ_LEN     = 100       # Characters per training sequence
BATCH_SIZE  = 128
EPOCHS      = 10
LR          = 0.001
GRAD_CLIP   = 5.0       # Gradient clipping threshold
VAL_SPLIT   = 0.1       # Fraction of data for validation

# Generation
TEMPERATURE = 0.8       # Sampling temperature (lower = more conservative)
GEN_LENGTH  = 120       # Max characters to generate

SEED        = 42
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
# 1. DATA LOADING & PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────

def clean_doha(text: str) -> str:
    """Remove verse markers and normalize whitespace."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"।।\s*\d+\s*।।", "।।", text)   # keep danda, drop number
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_corpus(csv_path: str):
    """
    Load the CSV, clean each doha, and return:
      - corpus      : single string (all dohas joined)
      - doha_list   : list of individual cleaned doha strings
    """
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    df = df.dropna(subset=["Doha"]).reset_index(drop=True)
    df["Doha_clean"] = df["Doha"].apply(clean_doha)

    doha_list = df["Doha_clean"].tolist()

    # Join with a special separator token between dohas
    corpus = "\n\n".join(doha_list)

    print(f"✅ Loaded {len(doha_list)} dohas")
    print(f"   Corpus length : {len(corpus):,} characters")
    return corpus, doha_list, df


def build_vocab(corpus: str):
    """
    Build character-level vocabulary from corpus.
    Returns:
      chars     : sorted list of unique characters
      char2idx  : dict mapping char → int
      idx2char  : dict mapping int → char
    """
    chars    = sorted(set(corpus))
    char2idx = {ch: i for i, ch in enumerate(chars)}
    idx2char = {i: ch for ch, i in char2idx.items()}
    print(f"   Vocabulary size: {len(chars)} unique characters")
    return chars, char2idx, idx2char


# ─────────────────────────────────────────────────────────────────────────────
# 2. DATASET
# ─────────────────────────────────────────────────────────────────────────────

class DohaCharDataset(Dataset):
    """
    Sliding-window character-level dataset.
    Each sample: (input_seq of length SEQ_LEN, target_seq shifted by 1)
    """
    def __init__(self, text: str, char2idx: dict, seq_len: int):
        self.seq_len  = seq_len
        self.char2idx = char2idx
        self.data     = [char2idx[ch] for ch in text if ch in char2idx]

    def __len__(self):
        return max(0, len(self.data) - self.seq_len)

    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx     : idx + self.seq_len], dtype=torch.long)
        y = torch.tensor(self.data[idx + 1 : idx + self.seq_len + 1], dtype=torch.long)
        return x, y


def split_corpus(corpus: str, val_split: float):
    """Split corpus string into train / val portions."""
    split = int(len(corpus) * (1 - val_split))
    return corpus[:split], corpus[split:]


# ─────────────────────────────────────────────────────────────────────────────
# 3. MODEL
# ─────────────────────────────────────────────────────────────────────────────

class CharLSTM(nn.Module):
    """
    Character-level LSTM language model.

    Architecture:
        Embedding → LSTM (stacked) → Dropout → Linear → logits

    The hidden state is carried across batches during training (stateful),
    giving the model longer context than a single SEQ_LEN window.
    """
    def __init__(self, vocab_size: int, embed_dim: int,
                 hidden_dim: int, num_layers: int, dropout: float):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding  = nn.Embedding(vocab_size, embed_dim)
        self.lstm       = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True
        )
        self.dropout    = nn.Dropout(dropout)
        self.fc         = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb    = self.embedding(x)                  # (B, T, E)
        out, hidden = self.lstm(emb, hidden)         # (B, T, H)
        out    = self.dropout(out)
        logits = self.fc(out)                        # (B, T, V)
        return logits, hidden

    def init_hidden(self, batch_size: int, device):
        h = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)
        c = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)
        return (h, c)

    @staticmethod
    def detach_hidden(hidden):
        """Detach hidden state from computation graph (truncated BPTT)."""
        return tuple(h.detach() for h in hidden)


# ─────────────────────────────────────────────────────────────────────────────
# 4. TRAINING
# ─────────────────────────────────────────────────────────────────────────────

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    hidden     = None

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        if hidden is None:
            hidden = model.init_hidden(x.size(0), device)
        else:
            # Re-init if batch size changed (last batch may be smaller)
            if hidden[0].size(1) != x.size(0):
                hidden = model.init_hidden(x.size(0), device)
            else:
                hidden = model.detach_hidden(hidden)

        optimizer.zero_grad()
        logits, hidden = model(x, hidden)           # (B, T, V)

        # Reshape for cross-entropy: (B*T, V) vs (B*T,)
        loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate_loss(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    hidden     = None

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        if hidden is None:
            hidden = model.init_hidden(x.size(0), device)
        else:
            if hidden[0].size(1) != x.size(0):
                hidden = model.init_hidden(x.size(0), device)
            else:
                hidden = model.detach_hidden(hidden)

        logits, hidden = model(x, hidden)
        loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        total_loss += loss.item()

    return total_loss / len(loader)


# ─────────────────────────────────────────────────────────────────────────────
# 5. EVALUATION METRICS
# ─────────────────────────────────────────────────────────────────────────────

def compute_perplexity(avg_loss: float) -> float:
    """
    Perplexity = exp(average cross-entropy loss).

    Lower is better. A perplexity of V (vocab size) means the model
    is as good as random guessing. Values near 1 = near-perfect prediction.

    For character-level models on Hindi, reasonable range: 2 – 10.
    """
    return math.exp(avg_loss)


@torch.no_grad()
def generate_text(model, seed_text: str, char2idx: dict, idx2char: dict,
                  length: int = GEN_LENGTH, temperature: float = TEMPERATURE,
                  device: str = DEVICE) -> str:
    """
    Auto-regressively generate characters from the model given a seed string.

    Temperature controls diversity:
      - Low  (0.2–0.5): conservative, repetitive but grammatical
      - High (0.8–1.2): creative but may drift
    """
    model.eval()

    # Encode seed
    input_ids = [char2idx[ch] for ch in seed_text if ch in char2idx]
    if not input_ids:
        input_ids = [random.randint(0, len(char2idx) - 1)]

    x      = torch.tensor([input_ids], dtype=torch.long).to(device)
    hidden = model.init_hidden(1, device)

    # Warm up hidden state on seed
    _, hidden = model(x, hidden)

    generated = seed_text
    x_t       = torch.tensor([[input_ids[-1]]], dtype=torch.long).to(device)

    for _ in range(length):
        logits, hidden = model(x_t, hidden)         # (1, 1, V)
        logits = logits[0, 0] / temperature
        probs  = torch.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, 1).item()
        next_char = idx2char[next_idx]
        generated += next_char
        x_t = torch.tensor([[next_idx]], dtype=torch.long).to(device)

    return generated


def compute_bleu(references: list, hypothesis: str) -> dict:
    """
    Compute character-level BLEU scores (BLEU-1 to BLEU-4).

    For doha generation we use CHARACTER-level BLEU because:
      - Dohas use archaic Braj vocabulary with few shared word tokens
      - Character n-grams capture script and phonetic similarity better
      - Word-level BLEU would be near-zero for any reasonable generation

    Args:
        references  : list of reference doha strings
        hypothesis  : generated doha string

    Returns:
        dict with bleu1..bleu4 scores and a smoothed corpus BLEU
    """
    smoother = SmoothingFunction().method1

    # Tokenize at CHARACTER level
    hyp_chars  = list(hypothesis)
    ref_chars  = [list(r) for r in references]

    scores = {}
    for n in range(1, 5):
        weights = tuple([1.0 / n] * n + [0.0] * (4 - n))
        scores[f"bleu{n}"] = sentence_bleu(
            ref_chars, hyp_chars,
            weights=weights,
            smoothing_function=smoother
        )

    # Corpus-level BLEU-4 across all references
    scores["bleu4_corpus"] = corpus_bleu(
        [[r] for r in ref_chars],
        [hyp_chars] * len(ref_chars),
        smoothing_function=smoother
    )

    return scores


def run_bleu_evaluation(model, doha_list: list, char2idx: dict,
                        idx2char: dict, n_samples: int = 20,
                        device: str = DEVICE) -> dict:
    """
    Generate n_samples dohas and compute BLEU against the full reference corpus.

    Strategy:
      - Take the first few characters of a random reference doha as seed
      - Generate the rest
      - Compare character-level n-gram overlap with all reference dohas
    """
    print(f"\n📊 Computing BLEU on {n_samples} generated samples...")

    all_bleu1, all_bleu2, all_bleu4 = [], [], []

    for _ in range(n_samples):
        ref_doha   = random.choice(doha_list)
        seed_len   = min(10, len(ref_doha) // 3)
        seed_text  = ref_doha[:seed_len]

        generated  = generate_text(model, seed_text, char2idx, idx2char,
                                   length=GEN_LENGTH, device=device)

        scores     = compute_bleu(doha_list, generated)
        all_bleu1.append(scores["bleu1"])
        all_bleu2.append(scores["bleu2"])
        all_bleu4.append(scores["bleu4"])

    results = {
        "bleu1_mean": np.mean(all_bleu1),
        "bleu2_mean": np.mean(all_bleu2),
        "bleu4_mean": np.mean(all_bleu4),
    }

    print(f"   BLEU-1 : {results['bleu1_mean']:.4f}  (character unigram overlap)")
    print(f"   BLEU-2 : {results['bleu2_mean']:.4f}  (character bigram overlap)")
    print(f"   BLEU-4 : {results['bleu4_mean']:.4f}  (character 4-gram overlap)")
    return results


# ─────────────────────────────────────────────────────────────────────────────
# 6. MAIN TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print(f"\n{'='*60}")
    print("Character-Level LSTM — Hindi Doha Training")
    print(f"Device : {DEVICE}")
    print(f"{'='*60}\n")

    # ── Load data ──────────────────────────────────────────
    corpus, doha_list, df = load_corpus(CSV_PATH)
    chars, char2idx, idx2char = build_vocab(corpus)
    vocab_size = len(chars)

    train_text, val_text = split_corpus(corpus, VAL_SPLIT)
    print(f"   Train chars : {len(train_text):,}")
    print(f"   Val chars   : {len(val_text):,}\n")

    train_dataset = DohaCharDataset(train_text, char2idx, SEQ_LEN)
    val_dataset   = DohaCharDataset(val_text,   char2idx, SEQ_LEN)

    train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                               shuffle=True,  drop_last=True, num_workers=0)
    val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                               shuffle=False, drop_last=True, num_workers=0)

    # ── Model ──────────────────────────────────────────────
    model = CharLSTM(
        vocab_size=vocab_size,
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT
    ).to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Model parameters: {total_params:,}\n")

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3
    )

    # ── Training loop ──────────────────────────────────────
    best_val_loss = float("inf")
    history = []

    print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>10} "
          f"{'Train PPL':>11} {'Val PPL':>9} {'Time':>7}")
    print("-" * 60)

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()

        train_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_loss   = evaluate_loss(model, val_loader, criterion, DEVICE)

        train_ppl  = compute_perplexity(train_loss)
        val_ppl    = compute_perplexity(val_loss)
        elapsed    = time.time() - t0

        prev_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]["lr"]
        
        # Log LR if scheduler reduced it this epoch
        if current_lr != prev_lr:
            print(f"         📉 LR reduced to {current_lr:.6f}")
        
        history.append({
            "epoch": epoch,
            "train_loss": train_loss, "val_loss": val_loss,
            "train_ppl":  train_ppl,  "val_ppl":  val_ppl,
            "lr": current_lr
        })

        print(f"{epoch:>6} {train_loss:>12.4f} {val_loss:>10.4f} "
              f"{train_ppl:>11.2f} {val_ppl:>9.2f} {elapsed:>6.1f}s")

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            ckpt_path = os.path.join(SAVE_DIR, "best_lstm_doha.pt")
            torch.save({
                "epoch":      epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "char2idx":   char2idx,
                "idx2char":   idx2char,
                "val_loss":   val_loss,
                "config": {
                    "vocab_size": vocab_size,
                    "embed_dim":  EMBED_DIM,
                    "hidden_dim": HIDDEN_DIM,
                    "num_layers": NUM_LAYERS,
                    "dropout":    DROPOUT,
                    "seq_len":    SEQ_LEN,
                }
            }, ckpt_path)
            print(f"         💾 Saved best model (val_ppl={val_ppl:.2f})")

        # Sample generation every 5 epochs
        if epoch % 5 == 0:
            seed = doha_list[0][:8] if doha_list else "मोर"
            sample = generate_text(model, seed, char2idx, idx2char,
                                   length=80, temperature=TEMPERATURE)
            print(f"\n  🎵 Sample [{seed!r}]: {sample}\n")

    # ── Final evaluation ───────────────────────────────────
    print(f"\n{'='*60}")
    print("FINAL EVALUATION")
    print(f"{'='*60}")

    # Load best checkpoint
    ckpt = torch.load(os.path.join(SAVE_DIR, "best_lstm_doha.pt"),
                      map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])

    final_val_loss = evaluate_loss(model, val_loader, criterion, DEVICE)
    final_ppl      = compute_perplexity(final_val_loss)

    print(f"\n  Best Val Loss  : {best_val_loss:.4f}")
    print(f"  Best Val PPL   : {compute_perplexity(best_val_loss):.2f}")

    # BLEU evaluation
    bleu_scores = run_bleu_evaluation(
        model, doha_list, char2idx, idx2char,
        n_samples=20, device=DEVICE
    )

    # ── Save training history ──────────────────────────────
    history_df = pd.DataFrame(history)
    history_path = os.path.join(SAVE_DIR, "training_history.csv")
    history_df.to_csv(history_path, index=False)
    print(f"\n  📈 Training history saved → {history_path}")

    # ── Summary ────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("SUMMARY")
    print(f"{'='*60}")
    print(f"  Vocab size     : {vocab_size}")
    print(f"  Best Val PPL   : {compute_perplexity(best_val_loss):.2f}")
    print(f"  BLEU-1         : {bleu_scores['bleu1_mean']:.4f}")
    print(f"  BLEU-2         : {bleu_scores['bleu2_mean']:.4f}")
    print(f"  BLEU-4         : {bleu_scores['bleu4_mean']:.4f}")
    print(f"{'='*60}\n")

    # ── Generate a few final samples ──────────────────────
    print("GENERATED DOHAS (temperature=0.8)")
    print("-" * 40)
    seeds = [d[:6] for d in doha_list[:3]] if len(doha_list) >= 3 else ["मोर"]
    for seed in seeds:
        out = generate_text(model, seed, char2idx, idx2char,
                            length=GEN_LENGTH, temperature=0.8)
        print(f"Seed: {seed!r}")
        print(f"  → {out}\n")


if __name__ == "__main__":
    main()


Character-Level LSTM — Hindi Doha Training
Device : cuda

✅ Loaded 8190 dohas
   Corpus length : 653,033 characters
   Vocabulary size: 134 unique characters
   Train chars : 587,729
   Val chars   : 65,304

   Model parameters: 899,078

 Epoch   Train Loss   Val Loss   Train PPL   Val PPL    Time
------------------------------------------------------------
     1       2.1141     1.9561        8.28      7.07  151.4s
         💾 Saved best model (val_ppl=7.07)
     2       1.8259     1.9762        6.21      7.22  156.8s
     3       1.7513     1.9940        5.76      7.35  156.7s
     4       1.7150     2.0038        5.56      7.42  157.2s
         📉 LR reduced to 0.000500
     5       1.6933     2.0227        5.44      7.56  156.9s

  🎵 Sample ['मोर पच्छ']: मोर पच्छाय। क्यों औ है बुख सूँ भरे, सुख की कहूँ अभीर ॥6॥

च्यामाता-सा-धरम की, करिये ज्ञा

     6       1.6599     2.0363        5.26      7.66  156.9s
     7       1.6489     2.0401        5.20      7.69  157.2s
     8       1.6416

In [9]:


print(f"\n{'='*60}")
print("FINAL EVALUATION")
print(f"{'='*60}")

# Load best checkpoint
ckpt = torch.load(os.path.join(SAVE_DIR, "/kaggle/working/best_lstm_doha.pt",
                  map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])

final_val_loss = evaluate_loss(model, val_loader, criterion, DEVICE)
final_ppl      = compute_perplexity(final_val_loss)

print(f"\n  Best Val Loss  : {best_val_loss:.4f}")
print(f"  Best Val PPL   : {compute_perplexity(best_val_loss):.2f}")

# BLEU evaluation
bleu_scores = run_bleu_evaluation(
    model, doha_list, char2idx, idx2char,
    n_samples=20, device=DEVICE
)

# ── Save training history ──────────────────────────────
history_df = pd.DataFrame(history)
history_path = os.path.join(SAVE_DIR, "training_history.csv")
history_df.to_csv(history_path, index=False)
print(f"\n  📈 Training history saved → {history_path}")

# ── Summary ────────────────────────────────────────────
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"  Vocab size     : {vocab_size}")
print(f"  Best Val PPL   : {compute_perplexity(best_val_loss):.2f}")
print(f"  BLEU-1         : {bleu_scores['bleu1_mean']:.4f}")
print(f"  BLEU-2         : {bleu_scores['bleu2_mean']:.4f}")
print(f"  BLEU-4         : {bleu_scores['bleu4_mean']:.4f}")
print(f"{'='*60}\n")

# ── Generate a few final samples ──────────────────────
print("GENERATED DOHAS (temperature=0.8)")
print("-" * 40)
seeds = [d[:6] for d in doha_list[:3]] if len(doha_list) >= 3 else ["मोर"]
for seed in seeds:
    out = generate_text(model, seed, char2idx, idx2char,
                        length=GEN_LENGTH, temperature=0.8)
    print(f"Seed: {seed!r}")
    print(f"  → {out}\n")

SyntaxError: '(' was never closed (4033147291.py, line 6)